# RAG CGU: fluxo completo, local (do banco à avaliação)
Execute as células **em ordem**. Tudo roda na sua máquina: criação do banco SQLite, ingestão dos PDFs, limpeza, enriquecimento,
embeddings, busca, resposta com LLM local e avaliação. A lógica está em `src/cgu_rag/`; aqui você só dispara e inspeciona.

**Antes de começar:** PDFs em `data/pdfs/` e, para a etapa 9, o servidor do LLM local rodando (README, passo 4).
Se mudar a configuração da etapa 1 depois de rodar, **reinicie o kernel** (o módulo legado lê as variáveis ao ser importado).

## 0. Ambiente
Localiza a raiz do projeto, ajusta o diretório de trabalho e confere as dependências.

In [ ]:
import os, sys, importlib.util
from pathlib import Path

# raiz do projeto = pasta que contém pyproject.toml (funciona se o notebook estiver em notebooks/ ou na raiz)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))          # permite rodar sem "pip install -e ."
print("Raiz:", RAIZ, "| Python:", sys.version.split()[0])

faltando = [m for m in ("fitz", "numpy", "requests", "sentence_transformers", "openai") if importlib.util.find_spec(m) is None]
if faltando:
    print("FALTANDO:", faltando)
    print('Instale com:  pip install -e ".[notebook]"   (no terminal, dentro do ambiente virtual) e reinicie o kernel')
else:
    print("Dependências OK")

## 1. Configuração
Caminhos, modelo de embeddings, LLM local e política de sigilo. Edite só esta célula.

In [ ]:
import os

os.environ["RAG_DB"] = "data/rag_cgu.db"                          # arquivo do banco (criado na etapa 2)
os.environ["RAG_EMB_MODEL"] = "intfloat/multilingual-e5-small"    # baseline; troque só depois do benchmark
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"          # Ollama local
os.environ["LLM_MODEL"] = "jurema-7b"                             # nome criado no "ollama create"
os.environ["SIGILO"] = "sigiloso"                                 # "sigiloso" = só endpoint local (falha fechada)

PASTA_PDFS = "data/pdfs"
GABARITO = "data/gabarito/perguntas.jsonl"

In [ ]:
%load_ext autoreload
%autoreload 2
from cgu_rag import config
from cgu_rag.legado import rag_cgu_pipeline as legado
from cgu_rag.busca import buscar, responder
print(config.settings)

## 2. Criar o banco
`legado.connect()` cria o arquivo SQLite e todo o esquema (`documents`, `chunks`, índice de texto `chunks_fts` com BM25, `meta`).
É idempotente: se o banco já existe, só abre (e migra colunas antigas).

In [ ]:
from pathlib import Path
Path(legado.DB_PATH).parent.mkdir(parents=True, exist_ok=True)

con = legado.connect()
print("Banco:", Path(legado.DB_PATH).resolve())
for (nome, tipo) in con.execute("SELECT name, type FROM sqlite_master WHERE type IN ('table','trigger') ORDER BY type, name"):
    print(f"  {tipo:8s} {nome}")

In [ ]:
# OPCIONAL: recomeçar do zero (apaga o banco). Descomente para usar.
# con.close(); Path(legado.DB_PATH).unlink(missing_ok=True); con = legado.connect(); print("banco recriado")

## 3. Ingestão dos PDFs
Extrai o texto com PyMuPDF, limpa, segmenta por seções e grava os trechos. PDFs já ingeridos e sem mudança são ignorados (hash SHA-256).

In [ ]:
pdfs = sorted(Path(PASTA_PDFS).glob("*.pdf"))
print(len(pdfs), "PDF(s) em", PASTA_PDFS)
assert pdfs, f"Coloque os PDFs em {PASTA_PDFS}/ e rode de novo"

for p in pdfs:
    try:
        legado.ingest(con, str(p))
    except Exception as e:
        print(f"! erro em {p.name}: {e}")
con.commit()

In [ ]:
for r in con.execute("SELECT id, titulo, n_paginas FROM documents"):
    c = con.execute("SELECT COUNT(*) n, SUM(tipo='secao') s FROM chunks WHERE doc_id=? AND tipo!='indice'", (r["id"],)).fetchone()
    print(f"doc {r['id']}: {r['titulo'][:60]!r} | {r['n_paginas']} págs | {c['n']} trechos | {c['s']} seções")

## 4. Relatório de qualidade da limpeza
Cobertura de páginas, tamanho dos trechos e ruído residual (cabeçalho/rodapé, hífens, caracteres de controle).

In [ ]:
for r in legado.quality_report(con):
    print(f"\ndoc {r['doc']}: {r['titulo']!r} | {r['paginas']} págs ({r['paginas_cobertas']} cobertas) | {r['chunks']} trechos "
          f"(mediana {r['tam_mediano']} car., {r['curtos_300']} < 300)")
    print("  ruído residual:", {k: v for k, v in r["ruido"].items()})

## 5. Enriquecimento (heurístico, sem LLM)
Gera resumo, palavras-chave, perguntas e contexto por regras. O enriquecimento **por LLM** do módulo legado usa provedores externos,
por isso não é usado aqui enquanto `SIGILO=sigiloso`.

In [ ]:
legado.enrich_pending(con, "heuristic")
con.commit()

## 6. Embeddings
Primeira execução baixa o modelo de embeddings (precisa de internet uma vez); depois fica em cache local. O banco trava o modelo usado: para trocar de modelo, limpe os embeddings.

In [ ]:
import time
t0 = time.time()
legado.embed_pending(con)
print(f"{time.time()-t0:.0f} s | modelo:", con.execute("SELECT v FROM meta WHERE k='emb_model'").fetchone()["v"])

In [ ]:
print("trechos:", con.execute("SELECT COUNT(*) FROM chunks WHERE tipo!='indice'").fetchone()[0],
      "| com vetor:", con.execute("SELECT COUNT(*) FROM chunks WHERE embedding IS NOT NULL").fetchone()[0])

## 7. Busca (BM25 + vetores, fusão RRF)

In [ ]:
def mostrar(q, k=5, denso=True):
    for score, r in buscar(con, q, k, denso):
        print(f"[{score:.4f}] {(r['nota_tecnica'] or r['doc_titulo'])[:55]} | {r['titulo'][:70]} (págs {r['pag_ini']}-{r['pag_fim']})")

mostrar("prazo para instauração de PAD")

Comparação rápida: só BM25 contra BM25 + vetores (use perguntas suas):

In [ ]:
q = "prazo para instauração de PAD"
print("--- só BM25"); mostrar(q, 5, denso=False)
print("--- BM25 + vetores"); mostrar(q, 5, denso=True)

## 8. Testar o LLM local
Se falhar, o servidor não está rodando ou o nome do modelo difere do criado no Ollama.

In [ ]:
from cgu_rag.llm_local import LLMLocal
llm = LLMLocal()
try:
    print(llm.chat([{"role": "user", "content": "Responda apenas: pronto."}], max_tokens=10))
    print(llm.medir("Explique em duas frases o que é processo administrativo disciplinar.", max_tokens=64))
except Exception as e:
    print("LLM indisponível:", repr(e))
    print("Confira: 'ollama list' mostra", config.settings.llm_model, "e o servidor está em", config.settings.llm_base_url)

## 9. Resposta com citação
Busca os trechos e pede ao LLM uma resposta **só com base neles**, citando documento e páginas.

In [ ]:
pergunta = "Quando a prescrição da pretensão punitiva começa a correr?"
out = responder(con, pergunta, k=4, llm=llm)
print(out["resposta"]); print()
for f in out["fontes"]: print("-", f)

## 10. Avaliação da recuperação (hit@k e MRR)
Crie `data/gabarito/perguntas.jsonl`, uma linha por pergunta: `{"id": "q01", "pergunta": "...", "esperado": "Nota Técnica nº ..."}`.
A célula cria um modelo com exemplos **fictícios** se o arquivo não existir; substitua pelo seu benchmark.

In [ ]:
import json
g = Path(GABARITO); g.parent.mkdir(parents=True, exist_ok=True)
if not g.exists():
    exemplo = [{"id": "q01", "pergunta": "Qual o prazo para instauração de PAD?", "esperado": "TROQUE-PELA-NOTA-TECNICA-ESPERADA"}]
    g.write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in exemplo), encoding="utf-8")
    print("modelo criado em", g, "-> edite antes de avaliar")

In [ ]:
from cgu_rag.avaliacao import avaliar_recuperacao
for denso in (False, True):
    for k in (1, 3, 5):
        print("BM25+vetores" if denso else "só BM25", k, avaliar_recuperacao(GABARITO, con, k=k, denso=denso))

Registre os números do baseline (A). Qualquer mudança futura (novo embedding, reranker, regex de metadados) deve ser comparada com eles.